In [1]:
import pandas as pd


In [3]:
df = pd.read_csv("datasets/cleaned_data.csv")

In [4]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'tenure_group', 'Additional_Service_Count', 'MCharges'],
      dtype='object')

In [5]:
x = df.drop(columns=['Churn'])
y = df['Churn']

In [13]:
x = x.drop(columns=["customerID", 'tenure_group', 'MCharges'])

In [14]:
print(x.shape)
print(y.shape)

(7043, 20)
(7043,)


In [15]:
x.dtypes

gender                       object
SeniorCitizen                 int64
Partner                      object
Dependents                   object
tenure                        int64
PhoneService                 object
MultipleLines                object
InternetService              object
OnlineSecurity               object
OnlineBackup                 object
DeviceProtection             object
TechSupport                  object
StreamingTV                  object
StreamingMovies              object
Contract                     object
PaperlessBilling             object
PaymentMethod                object
MonthlyCharges              float64
TotalCharges                float64
Additional_Service_Count      int64
dtype: object

In [16]:
x.select_dtypes(include='object').columns

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [17]:
for col in x.select_dtypes(include='object').columns:
    print(col, ":", x[col].unique())

gender : ['Female' 'Male']
Partner : ['Yes' 'No']
Dependents : ['No' 'Yes']
PhoneService : ['No' 'Yes']
MultipleLines : ['No phone service' 'No' 'Yes']
InternetService : ['DSL' 'Fiber optic' 'No']
OnlineSecurity : ['No' 'Yes' 'No internet service']
OnlineBackup : ['Yes' 'No' 'No internet service']
DeviceProtection : ['No' 'Yes' 'No internet service']
TechSupport : ['No' 'Yes' 'No internet service']
StreamingTV : ['No' 'Yes' 'No internet service']
StreamingMovies : ['No' 'Yes' 'No internet service']
Contract : ['Month-to-month' 'One year' 'Two year']
PaperlessBilling : ['Yes' 'No']
PaymentMethod : ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


In [20]:
binary_cols = []

for col in x.select_dtypes(include="object").columns:
    if x[col].nunique() == 2:
        binary_cols.append(col)

print(binary_cols)

multi_cols = []
for col in x.select_dtypes(include="object").columns:
    if x[col].nunique() > 2:
        multi_cols.append(col)

print(multi_cols)


['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [23]:
x[binary_cols] = x[binary_cols].replace({
    'Yes':1,
    'No':0,
    'Male':1,
    'Female':0,
})

x[binary_cols]

,gender,Partner,Dependents,PhoneService,PaperlessBilling
0,0,1,0,0,1
1,1,0,0,1,0
2,1,0,0,1,1
3,1,0,0,0,0
4,0,0,0,1,1
...,...,...,...,...,...
7038,1,1,1,1,1
7039,0,1,1,1,1
7040,0,1,1,0,1
7041,1,1,0,1,1


In [24]:
x = pd.get_dummies(x,
    columns=multi_cols,
    drop_first=True,
    dtype=int)

In [25]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   PaperlessBilling                       7043 non-null   int64  
 7   MonthlyCharges                         7043 non-null   float64
 8   TotalCharges                           7043 non-null   float64
 9   Additional_Service_Count               7043 non-null   int64  
 10  MultipleLines_No phone service         7043 non-null   int64  
 11  Mult

In [27]:
x.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Additional_Service_Count,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,1,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,2,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,2,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,3,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,0,...,0,0,0,0,0,0,0,0,1,0


In [29]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y, 
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [34]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numerical_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Additional_Service_Count"
]
scaler.fit(x_train[numerical_cols])

StandardScaler()

In [35]:

x_train[numerical_cols] = scaler.transform(x_train[numerical_cols])
x_test[numerical_cols] = scaler.transform(x_test[numerical_cols])

In [36]:
x_train_scaled[:5]

array([[ 0.99433624, -0.44177295, -0.96923413, -0.65155653,  0.10237124,
        -3.01309011, -1.20265302, -0.52197565, -0.2622572 ,  0.50793527,
         3.01309011, -0.85833837, -0.88769579, -0.52408075, -0.52408075,
        -0.63611103, -0.52408075, -0.73554636, -0.52408075,  1.37557156,
        -0.52408075, -0.64327425, -0.52408075,  1.25216312, -0.52408075,
         1.24796703, -0.51278214, -0.56382155, -0.52380561,  1.40690298,
        -0.54384572],
       [ 0.99433624, -0.44177295,  1.03174245,  1.53478624, -0.71174346,
         0.3318852 , -1.20265302,  0.33747781, -0.50363479, -0.57052989,
        -0.3318852 , -0.85833837,  1.12651205, -0.52408075, -0.52408075,
         1.57205259, -0.52408075, -0.73554636, -0.52408075, -0.72697054,
        -0.52408075, -0.64327425, -0.52408075, -0.79861799, -0.52408075,
        -0.80130322, -0.51278214, -0.56382155, -0.52380561, -0.71078107,
         1.83875676],
       [ 0.99433624, -0.44177295,  1.03174245,  1.53478624, -0.79315493,
       

In [38]:
import pandas as pd

x_train_scaled = pd.DataFrame(
    x_train_scaled,
    columns=x_train.columns,
    index=x_train.index
)

x_test_scaled = pd.DataFrame(
    x_test_scaled,
    columns=x_test.columns,
    index=x_test.index
)

In [39]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

In [40]:
model.fit(x_train_scaled, y_train)

LogisticRegression()

In [42]:
y_pred = model.predict(x_test_scaled)
print(y_pred[:20])

['No' 'Yes' 'No' 'No' 'No' 'Yes' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'Yes'
 'No' 'No' 'No' 'Yes' 'No' 'No']


In [45]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy : {accuracy*100:.2f}")

Accuracy : 80.70


In [46]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score")
print(roc_auc_score(y_test, model.predict_proba(x_test_scaled)[:, 1]))





Confusion Matrix
[[925 110]
 [162 212]]

Classification Report
              precision    recall  f1-score   support

          No       0.85      0.89      0.87      1035
         Yes       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


ROC-AUC Score
0.8417964814384252


In [ ]:
# Random Forest

